In [24]:
# Install zstd first
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd

# Now install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama in the background
!nohup ollama serve > /tmp/ollama.log 2>&1 &
!sleep 8

# Pull a small model that fits Colab's free RAM
!ollama pull llama3.2

# Install the Python OpenAI client
!pip install -q openai

# Verify Ollama is up
!ollama list

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

NAME               ID              SIZE      MODIFIED      
llama3.2:latest    a80c4f17acd5    2.0 GB    6 seconds ago    


In [25]:
%%writefile interview.py
from __future__ import annotations

import argparse
import csv
import io
import json
import math
import os
import re
import sys
import time
from collections import Counter
from dataclasses import dataclass, field, asdict
from pathlib import Path

# ===========================================================================
#  CONFIG — Ollama inside Colab
# ===========================================================================
RUN_BACKEND   = "openai"
RUN_MODEL     = "llama3.2"
RUN_BASE_URL  = "http://localhost:11434/v1"
OPENAI_API_KEY = "ollama"
RUN_TECHNICAL = 5
RUN_BEHAVIORAL = 3
RUN_FORMAT    = "md"
RUN_OUT       = None
# ===========================================================================


SAMPLE_JD = """Data Analyst Intern

We are looking for a Data Analyst Intern to join our analytics team for 3 months.
You will clean and analyze datasets, write SQL queries, build dashboards, and present
findings to product managers.

Requirements:
- Working knowledge of Python (pandas, numpy) and SQL
- Basic statistics
- Experience with a visualization tool such as Tableau or Power BI
- Familiarity with Git
- Good communication and willingness to learn
"""

SAMPLE_PROFILES = [
    {
        "name": "Aisha Khan",
        "education": "BS Computer Science, 3rd year",
        "skills": ["Python", "pandas", "Matplotlib", "Excel"],
        "projects": [
            "Analysed a public COVID-19 dataset with pandas and visualized trends with Matplotlib",
            "Built a Flask web app that tracks personal expenses",
        ],
        "experience": "Teaching assistant for an introductory programming course",
    },
    {
        "name": "Omar Siddiqui",
        "education": "BS Statistics, final year",
        "skills": ["R programming", "SQL", "Tableau", "statistics"],
        "projects": ["Customer churn analysis using SQL queries and a Tableau dashboard"],
        "experience": "Part-time data entry and reporting at a retail company",
    },
]

SAMPLE_BANK_CSV = """category,role,skill,difficulty,question
technical,Data Analyst,sql,easy,What is the difference between INNER JOIN and LEFT JOIN? When would you use each?
technical,Data Analyst,sql,medium,How would you find duplicate rows in a table and remove them?
technical,Data Analyst,pandas,easy,How do you handle missing values in a pandas DataFrame?
technical,Data Analyst,statistics,easy,Explain the difference between mean and median and when the median is more useful.
technical,Data Analyst,data visualization,easy,How do you decide between a bar chart and a line chart?
technical,Software Engineer,python,easy,What is the difference between a list and a tuple in Python?
technical,Software Engineer,python,medium,Explain how Python decorators work and give a use case.
technical,Software Engineer,git,easy,What is the difference between git merge and git rebase?
technical,Software Engineer,rest apis,easy,What do the HTTP methods GET and POST mean and how do they differ?
technical,Software Engineer,data structures,medium,When would you choose a hash map over an array?
technical,Software Engineer,testing,easy,What makes a good unit test?
technical,Machine Learning Engineer,machine learning,easy,What is overfitting and how can you reduce it?
technical,Machine Learning Engineer,scikit-learn,medium,How do you evaluate a classifier on an imbalanced dataset?
technical,Web Developer,javascript,easy,What is the difference between let and const and var?
technical,Web Developer,react,medium,How does the useEffect hook work and what is the dependency array for?
behavioral,General,teamwork,easy,Tell me about a time you worked in a team on a project. What was your role?
behavioral,General,learning,easy,Describe something technical you taught yourself recently. How did you go about it?
behavioral,General,feedback,medium,Tell me about a time you received critical feedback. What did you do with it?
behavioral,General,deadlines,medium,Describe a time you had too much to do in too little time. How did you prioritize?
behavioral,General,ownership,medium,Tell me about a project you are proud of and what you personally contributed.
"""


@dataclass
class InternProfile:
    name: str = "Candidate"
    education: str = ""
    skills: list = field(default_factory=list)
    projects: list = field(default_factory=list)
    experience: str = ""

    @classmethod
    def from_dict(cls, d: dict) -> "InternProfile":
        return cls(d.get("name", "Candidate"), d.get("education", ""),
                   list(d.get("skills", [])), list(d.get("projects", [])),
                   d.get("experience", ""))

    def summary(self) -> str:
        parts = [f"Name: {self.name}"]
        if self.education: parts.append(f"Education: {self.education}")
        if self.skills: parts.append("Skills: " + ", ".join(self.skills))
        if self.projects: parts.append("Projects:\n" + "\n".join(f"- {p}" for p in self.projects))
        if self.experience: parts.append(f"Experience: {self.experience}")
        return "\n".join(parts)


@dataclass
class Question:
    category: str
    question: str
    skill: str = ""
    difficulty: str = "medium"
    good_answer_points: list = field(default_factory=list)


@dataclass
class SkillAnalysis:
    required: list
    candidate: list
    matched: list
    gaps: list


@dataclass
class QuestionSet:
    candidate: str
    role: str
    backend: str
    analysis: SkillAnalysis
    questions: list

    def to_markdown(self) -> str:
        a = self.analysis
        out = [f"# Interview Questions: {self.candidate}",
               f"**Role:** {self.role}   |   **Backend:** {self.backend}", "",
               f"- **Matched skills:** {', '.join(a.matched) or 'none'}",
               f"- **Skill gaps:** {', '.join(a.gaps) or 'none'}", ""]
        for qtype in ("technical", "behavioral"):
            group = [q for q in self.questions if q.category == qtype]
            if not group: continue
            out.append(f"## {qtype.title()} questions")
            for i, q in enumerate(group, 1):
                tag = ", ".join(x for x in (q.skill, q.difficulty) if x)
                out.append(f"{i}. {q.question}  _({tag})_")
                for p in q.good_answer_points:
                    out.append(f"   - {p}")
            out.append("")
        return "\n".join(out)


SKILLS = {
    "python": ["python"], "java": ["java"], "javascript": ["javascript", "js"],
    "typescript": ["typescript"], "c++": ["c++"], "c#": ["c#"],
    "sql": ["sql", "mysql", "postgresql", "postgres", "sqlite"],
    "r": ["r programming"], "html/css": ["html", "css", "html5", "css3"],
    "react": ["react", "reactjs", "react.js"], "node.js": ["node.js", "nodejs", "node"],
    "django": ["django"], "flask": ["flask"], "fastapi": ["fastapi"],
    "rest apis": ["rest", "restful", "rest api", "rest apis", "api", "apis"],
    "pandas": ["pandas"], "numpy": ["numpy"], "scikit-learn": ["scikit-learn", "sklearn"],
    "machine learning": ["machine learning", "ml"],
    "deep learning": ["deep learning", "pytorch", "tensorflow", "keras"],
    "nlp": ["nlp", "natural language processing"],
    "data visualization": ["tableau", "power bi", "matplotlib", "seaborn",
                           "data visualization", "dashboards", "dashboard"],
    "statistics": ["statistics", "statistical"], "excel": ["excel"],
    "data analysis": ["data analysis", "data analytics"],
    "git": ["git", "github", "gitlab"], "docker": ["docker"],
    "kubernetes": ["kubernetes", "k8s"], "linux": ["linux", "unix", "bash"],
    "aws": ["aws", "amazon web services"], "azure": ["azure"],
    "gcp": ["gcp", "google cloud"], "ci/cd": ["ci/cd", "jenkins", "github actions"],
    "testing": ["unit testing", "pytest", "junit", "test automation", "testing"],
    "mongodb": ["mongodb"], "data structures": ["data structures", "algorithms"],
    "oop": ["oop", "object-oriented", "object oriented"], "agile": ["agile", "scrum"],
}

_SKILL_RX = {
    canon: re.compile(r"(?<![\w+#.])(?:%s)(?![\w#]|\+\+)" %
                      "|".join(re.escape(a) for a in sorted(set(al + [canon]), key=len, reverse=True)),
                      re.IGNORECASE)
    for canon, al in SKILLS.items()
}


def extract_skills(text: str) -> list:
    return [c for c, rx in _SKILL_RX.items() if rx.search(text or "")]


def analyze_skills(jd: str, p: InternProfile) -> SkillAnalysis:
    required = extract_skills(jd)
    candidate = extract_skills(" , ".join([*p.skills, *p.projects, p.education, p.experience]))
    return SkillAnalysis(required, candidate,
                         [s for s in required if s in candidate],
                         [s for s in required if s not in candidate])


def guess_role(jd: str) -> str:
    for line in jd.splitlines():
        line = line.strip(" #*\t")
        if line: return line[:100]
    return "Intern"


_STOP = set("a an the and or of to in on for with is are was were you your how what why when do does did "
            "can would could that this it as at by be about me i we us".split())


def tokenize(text: str) -> list:
    return [t for t in (x.strip(".") for x in re.findall(r"[a-z0-9+#.]+", text.lower()))
            if t and t not in _STOP]


@dataclass
class BankEntry:
    category: str
    question: str
    skill: str = ""
    role: str = ""
    difficulty: str = "medium"


class QuestionBank:
    def __init__(self, entries: list):
        self.entries = entries
        counts = [Counter(tokenize(f"{e.question} {e.skill} {e.role}")) for e in entries]
        df = Counter()
        for c in counts: df.update(c.keys())
        n = len(entries)
        self.idf = {t: math.log((1 + n) / (1 + d)) + 1 for t, d in df.items()}
        self.vecs = [self._weigh(c) for c in counts]

    @classmethod
    def from_rows(cls, rows: list) -> "QuestionBank":
        entries = []
        for i, r in enumerate(rows, 1):
            q, cat = (r.get("question") or "").strip(), (r.get("category") or "").strip().lower()
            if not q or cat not in ("technical", "behavioral"):
                raise ValueError(f"row {i}: needs 'question' and category technical|behavioral")
            entries.append(BankEntry(cat, q, (r.get("skill") or "").strip().lower(),
                                     (r.get("role") or "").strip(),
                                     (r.get("difficulty") or "medium").strip().lower()))
        return cls(entries)

    @classmethod
    def sample(cls) -> "QuestionBank":
        return cls.from_rows(list(csv.DictReader(io.StringIO(SAMPLE_BANK_CSV))))

    def _weigh(self, counts: Counter) -> dict:
        v = {t: c * self.idf.get(t, 1.0) for t, c in counts.items()}
        norm = math.sqrt(sum(x * x for x in v.values())) or 1.0
        return {t: x / norm for t, x in v.items()}

    def search(self, query: str, k: int, category: str, skills: list) -> list:
        qv, wanted, scored = self._weigh(Counter(tokenize(query))), set(skills), []
        for e, vec in zip(self.entries, self.vecs):
            if e.category != category: continue
            score = sum(w * vec.get(t, 0.0) for t, w in qv.items()) + (0.5 if e.skill in wanted else 0.0)
            if score > 0: scored.append((score, e))
        scored.sort(key=lambda s: -s[0])
        return [e for _, e in scored[:k]]


SYSTEM_PROMPT = ("You are an experienced engineering manager who interviews interns. "
                 "Write fair, role-specific questions. Never ask about age, gender, family, "
                 "religion, nationality, health or other protected characteristics. "
                 "Reply with one JSON object only.")


def build_prompt(jd, profile, analysis, examples, n_tech, n_beh) -> str:
    ex = "\n".join(f"- [{e.category}/{e.skill or 'general'}] {e.question}" for e in examples) or "(none)"
    return f"""JOB DESCRIPTION
{jd.strip()}

CANDIDATE PROFILE
{profile.summary()}

SKILLS: matched = {', '.join(analysis.matched) or 'none'}; gaps = {', '.join(analysis.gaps) or 'none'}

EXAMPLE QUESTIONS (style only; do not copy)
{ex}

Write exactly {n_tech} technical and {n_beh} behavioral interview questions.
Return JSON: {{"questions":[{{"type":"technical|behavioral","question":"...","skill":"...",
"difficulty":"easy|medium|hard","good_answer_points":["...","..."]}}]}}"""


def parse_json(raw: str) -> dict:
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw.strip(), flags=re.IGNORECASE)
    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        a = text.find("{")
        if a == -1: raise ValueError("no JSON object in model output")
        data, _ = json.JSONDecoder().raw_decode(text, a)
    return {"questions": data} if isinstance(data, list) else data


def validate_item(item) -> Question | None:
    if not isinstance(item, dict): return None
    qtype = str(item.get("type", "")).strip().lower()
    text = str(item.get("question", "")).strip()
    if qtype not in ("technical", "behavioral") or len(text) < 15 or not text.endswith(("?", ".")):
        return None
    diff = str(item.get("difficulty", "medium")).strip().lower()
    pts = item.get("good_answer_points", [])
    pts = pts if isinstance(pts, list) else [str(pts)]
    return Question(qtype, text, str(item.get("skill", "")).strip().lower(),
                    diff if diff in ("easy", "medium", "hard") else "medium",
                    [str(p).strip() for p in pts if str(p).strip()][:5])


class OpenAIBackend:
    name = "openai"
    def __init__(self, model, base_url=None, api_key=None):
        from openai import OpenAI
        self.model = model
        self.client = OpenAI(api_key=api_key or "none", base_url=base_url, timeout=180)

    def generate(self, jd, profile, analysis, examples, n_tech, n_beh) -> list:
        msgs = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": build_prompt(jd, profile, analysis, examples, n_tech, n_beh)}]
        last = None
        for attempt in range(3):
            try:
                r = self.client.chat.completions.create(model=self.model, messages=msgs, temperature=0.7)
                return parse_json(r.choices[0].message.content or "").get("questions", [])
            except Exception as e:
                last = e
                time.sleep(2 ** attempt)
        raise RuntimeError(f"LLM request failed: {last}")


class QuestionGenerator:
    def __init__(self, bank, backend):
        self.bank, self.backend = bank, backend

    def generate(self, jd, profile, n_tech=5, n_beh=3) -> QuestionSet:
        analysis = analyze_skills(jd, profile)
        focus = analysis.required or analysis.candidate
        examples = (self.bank.search(jd, 5, "technical", focus) +
                    self.bank.search(jd, 3, "behavioral", []))
        accepted, need = [], {"technical": n_tech, "behavioral": n_beh}
        count = lambda t: sum(q.category == t for q in accepted)
        for item in self.backend.generate(jd, profile, analysis, examples, n_tech, n_beh):
            q = validate_item(item)
            if q and count(q.category) < need[q.category]:
                accepted.append(q)
        accepted.sort(key=lambda q: q.category != "technical")
        return QuestionSet(profile.name, guess_role(jd), self.backend.name, analysis, accepted)


def run():
    print("=" * 70)
    print(f"Backend : {RUN_BACKEND}")
    print(f"Model   : {RUN_MODEL}")
    print(f"Base URL: {RUN_BASE_URL}")
    print("=" * 70)
    bank = QuestionBank.sample()
    jd = SAMPLE_JD
    profiles = [InternProfile.from_dict(d) for d in SAMPLE_PROFILES]
    backend = OpenAIBackend(RUN_MODEL, RUN_BASE_URL, OPENAI_API_KEY)
    generator = QuestionGenerator(bank, backend)
    results = []
    for p in profiles:
        print(f"\nGenerating questions for: {p.name} ...")
        results.append(generator.generate(jd, p, RUN_TECHNICAL, RUN_BEHAVIORAL))
    text = "\n\n---\n\n".join(r.to_markdown() for r in results)
    if RUN_OUT:
        Path(RUN_OUT).write_text(text, encoding="utf-8")
        print(f"\nWrote {RUN_OUT}")
    else:
        print("\n" + text)
    return results


if __name__ == "__main__":
    run()

Writing interview.py


In [26]:
!python interview.py

Backend : openai
Model   : llama3.2
Base URL: http://localhost:11434/v1

Generating questions for: Aisha Khan ...

Generating questions for: Omar Siddiqui ...

# Interview Questions: Aisha Khan
**Role:** Data Analyst Intern   |   **Backend:** openai

- **Matched skills:** python, pandas, data visualization
- **Skill gaps:** sql, numpy, statistics, git

## Behavioral questions
1. Tell me about a time you worked in a team on a project. What was your role?  _(teamwork, easy)_
   - Describe your experience working with a team, and how you contributed to the project. Emphasize your communication and collaboration skills.


---

# Interview Questions: Omar Siddiqui
**Role:** Data Analyst Intern   |   **Backend:** openai

- **Matched skills:** sql, data visualization, statistics
- **Skill gaps:** python, pandas, numpy, git

## Technical questions
1. What is the difference between INNER JOIN and LEFT JOIN? When would you use each?  _(sql, easy)_
2. How do you decide between a bar chart and a l